In [7]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from IPython.display import display

print("APM 1111 - FA10\nBorja, Rachelle Erika D.")

# Clean table Format
def clean_table(df):
    df = df.copy()

    # Force ALL numeric values to 4 decimal places
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].apply(lambda x: f"{x:.4f}")

    styles = [
        {'selector': 'th', 'props': [('text-align', 'center'), ('font-weight', 'bold')]},
        {'selector': 'td', 'props': [('text-align', 'center')]}
    ]

    # First column left aligned
    styles.append({
        'selector': 'td:nth-child(1)',
        'props': [('text-align', 'left')]
    })

    return display(
        df.style
        .hide(axis="index")
        .set_table_styles(styles)
    )

# Problem 1
print("\n" + "-" * 80 + "\n")
print("PROBLEM 1:\n")

X = np.array([6,5,8,8,7,6,10,4,9,7])
Y = np.array([8,7,7,10,5,8,10,6,8,6])

n = len(X)

x_bar = np.mean(X)
y_bar = np.mean(Y)

print(f"Mean of X = {x_bar}")
print(f"Mean of Y = {y_bar}")

# deviations
dx = X - x_bar
dy = Y - y_bar

print("\nDeviations (X - mean):", dx)
print("Deviations (Y - mean):", dy)

# variances
sX = np.sqrt(np.sum(dx**2)/(n-1))
sY = np.sqrt(np.sum(dy**2)/(n-1))

print(f"\nStandard deviation of X = {sX:.4f}")
print(f"Standard deviation of Y = {sY:.4f}")

# correlation
r1 = np.sum(dx*dy) / np.sqrt(np.sum(dx**2)*np.sum(dy**2))
print(f"\nCorrelation r = {r1:.4f}")

# standard errors
s_Y_X = sY * np.sqrt(1 - r1**2)
s_X_Y = sX * np.sqrt(1 - r1**2)

print(f"\ns_Y·X = {s_Y_X:.4f}")
print(f"s_X·Y = {s_X_Y:.4f}")

clean_table(pd.DataFrame({
    "Measure": ["s_Y·X", "s_X·Y"],
    "Value": [s_Y_X, s_X_Y]
}))
print("\n" + "-" * 80 + "\n")

# Problem 2
print("PROBLEM 2:\n")

temperature = np.array([0,20,40,60,80,100,120,140,160,180,200,220,240,260,280,300,320,340,360])
pressure = np.array([0.0002,0.0012,0.0060,0.0300,0.0900,0.2700,0.7500,1.8500,
                     4.2000,8.8000,17.3000,32.1000,57.0000,96.0000,157.0000,
                     247.0000,376.0000,558.0000,806.0000])

r2 = np.corrcoef(temperature, pressure)[0,1]
print(f"Correlation r(X,Y) = {r2:.4f}")

X_prime = 2*temperature + 6
Y_prime = 3*pressure - 15

r2_prime = np.corrcoef(X_prime, Y_prime)[0,1]
print(f"Correlation r(X',Y') = {r2_prime:.4f}")

clean_table(pd.DataFrame({
    "Correlation": ["r(X,Y)", "r(X',Y')"],
    "Value": [r2, r2_prime]
}))
print("\n" + "-" * 80 + "\n")

# Problem 3
print("PROBLEM 3:\n")

flights = pd.read_csv(
    "https://raw.githubusercontent.com/byuidatascience/data4python4ds/master/data-raw/flights/flights.csv"
)

df = flights[['dep_delay','arr_delay']].dropna()

r3 = df['dep_delay'].corr(df['arr_delay'])
print(f"Correlation r(dep_delay, arr_delay) = {r3:.4f}")

X3_prime = 2*df['dep_delay'] + 10
Y3_prime = 0.5*df['arr_delay'] - 5

r3_prime = X3_prime.corr(Y3_prime)
print(f"Correlation r' = {r3_prime:.4f}")

clean_table(pd.DataFrame({
    "Correlation": [
        "r(dep_delay, arr_delay)",
        "r' (transformed)"
    ],
    "Value": [r3, r3_prime]
}))
print("\n" + "-" * 80 + "\n")

# Problem 4
print("PROBLEM 4:")

df_reg = flights[['arr_delay','dep_delay','air_time','distance','hour','carrier']].dropna()

df_reg['X1'] = (df_reg['dep_delay'] > 0).astype(int)
df_reg['X2'] = df_reg['dep_delay']
df_reg['X3'] = df_reg['air_time']
df_reg['X4'] = df_reg['distance']
df_reg['X5'] = df_reg['hour']

major_carriers = ['AA','DL','UA','WN']
df_reg['X6'] = df_reg['carrier'].isin(major_carriers).astype(int)

X_reg = df_reg[['X1','X2','X3','X4','X5','X6']]
Y_reg = df_reg['arr_delay']

X_reg = sm.add_constant(X_reg)

print("\nFitting OLS model...")
model = sm.OLS(Y_reg, X_reg).fit()

print("\nKey values:")
print(f"Coefficient X2 = {model.params['X2']:.4f}")
print(f"R^2 = {model.rsquared:.4f}")
print(f"F-test p-value = {model.f_pvalue}")

clean_table(pd.DataFrame({
    "Metric": ["Coefficient of X2", "R^2", "F-test p-value"],
    "Value": [model.params['X2'], model.rsquared, model.f_pvalue]
}))
print("\n" + "-" * 80 + "\n")

# Regression Tables
print("REGRESSION TABLES:\n")

# coefficients
coef_table = pd.DataFrame({
    "Variable": model.params.index,
    "Coefficient": model.params.values,
    "Std Error": model.bse.values,
    "t-value": model.tvalues.values,
    "p-value": model.pvalues.values
})
clean_table(coef_table)
print("\n")

# confidence intervals
conf = model.conf_int()
conf_table = pd.DataFrame({
    "Variable": conf.index,
    "Lower (0.025)": conf[0].values,
    "Upper (0.975)": conf[1].values
})
clean_table(conf_table)
print("\n")
# diagnostics (FIXED)
from statsmodels.stats.stattools import durbin_watson, omni_normtest, jarque_bera

dw = durbin_watson(model.resid)
omni_stat, omni_p = omni_normtest(model.resid)
jb_stat, jb_p, _, _ = jarque_bera(model.resid)

clean_table(pd.DataFrame({
    "Metric": [
        "Durbin-Watson",
        "Omnibus",
        "Prob(Omnibus)",
        "Jarque-Bera",
        "Prob(JB)",
        "Skew",
        "Kurtosis"
    ],
    "Value": [
        dw, omni_stat, omni_p,
        jb_stat, jb_p,
        model.resid.skew(),
        model.resid.kurtosis()
    ]
}))
print("\n" + "-" * 80 + "\n")

# Final Summary
print("FINAL SUMMARY:\n")

clean_table(pd.DataFrame({
    "Problem": [
        "1: s_Y·X", "1: s_X·Y",
        "2: r(X,Y)", "2: r(X',Y')",
        "3: r(dep,arr)", "3: r'(transformed)",
        "4: coef X2", "4: R^2", "4: F-test p-value"
    ],
    "Result": [
        s_Y_X, s_X_Y,
        r2, r2_prime,
        r3, r3_prime,
        model.params['X2'],
        model.rsquared,
        model.f_pvalue
    ]
}))
print("\n" + "-" * 80 + "\n")

APM 1111 - FA10
Borja, Rachelle Erika D.

--------------------------------------------------------------------------------

PROBLEM 1:

Mean of X = 7.0
Mean of Y = 7.5

Deviations (X - mean): [-1. -2.  1.  1.  0. -1.  3. -3.  2.  0.]
Deviations (Y - mean): [ 0.5 -0.5 -0.5  2.5 -2.5  0.5  2.5 -1.5  0.5 -1.5]

Standard deviation of X = 1.8257
Standard deviation of Y = 1.6499

Correlation r = 0.5533

s_Y·X = 1.3744
s_X·Y = 1.5208


Measure,Value
s_Y·X,1.3744
s_X·Y,1.5208



--------------------------------------------------------------------------------

PROBLEM 2:

Correlation r(X,Y) = 0.7578
Correlation r(X',Y') = 0.7578


Correlation,Value
"r(X,Y)",0.7578
"r(X',Y')",0.7578



--------------------------------------------------------------------------------

PROBLEM 3:

Correlation r(dep_delay, arr_delay) = 0.9148
Correlation r' = 0.9148


...

Variable,Lower (0.025),Upper (0.975)
const,-14.7048,-14.3115
X1,1.3461,1.6073
X2,1.0088,1.0120
X3,0.6848,0.6931
X4,-0.0890,-0.0879
X5,-0.0969,-0.0733
X6,-4.0646,-3.8333


Metric,Value
Durbin-Watson,1.4555
Omnibus,117827.2147
Prob(Omnibus),0.0000
Jarque-Bera,781162.7188
Prob(JB),0.0000
Skew,1.5801
Kurtosis,6.8766



--------------------------------------------------------------------------------

FINAL SUMMARY:



Problem,Result
1: s_Y·X,1.3744
1: s_X·Y,1.5208
"2: r(X,Y)",0.7578
"2: r(X',Y')",0.7578
"3: r(dep,arr)",0.9148
3: r'(transformed),0.9148
4: coef X2,1.0104
4: R^2,0.8792
4: F-test p-value,0.0000



--------------------------------------------------------------------------------

